# COMS W2132 Intermediate Computing in Python
## Arrays, Stacks, and Queues

**Date**: February 11, 2026\
Daniel Bauer &lt;bauer@cs.columbia.edu&gt; (original notes by Jan Janak)

**Reading**: Data Structures and Algorithms in Python, Chapter 6

---

## Review of Built-in Python Sequence Types

* Python's three built-in sequence types are lists, tuples, and strings.
* Each supports indexing, i.e., access to individual elements using the syntax `seq[i]`.
* **Important: indexing is guaranteed to be $O(1)$.**
* All three sequence types are implemented using the same internal data structure called **array**.

_We will use the built-in sequence types to build more complex data structures._

In [17]:
# Lists, tuples, and strings can be created using the literal syntax
lst = [1, 2, 3]
tpl = (4, 5, 6)
txt = "SAMPLE"

# And we can access and print individual elements using the integer indexing syntax
print(lst[1])
print(tpl[1])
print(txt[1])

2
5
A


Lists, tuples, and strings internally store data in arrays. An array is a set of memory locations addressed using consecutive indices.

A memory location can store a value (e.g., a number or string character). This is called a compact array:

<img src="https://janakj.org/w3132/images/compact-array.png" width=300/>

Python uses compact arrays to represent strings.

A memory location can also store a reference to an object. This is called a referential array:

<img src="https://janakj.org/w3132/images/referential-array.png" width=600/>

Referential arrays are used to represent lists and tuples. The above diagram represents the list

```python
names = ["Rene", "Joseph", "Janet", "Jonas", "Helen", "Virginia"]
```

### Asymptotic Running Times of Elementary Operations on Arrays
* Space (memory) used by arrays: $O(n)$, where $n$ is the number of elements
* Integer indexing is always $O(1)$.
* Insertion in the worst case is $O(n)$. The worst case is inserting an element at index 0. In that case, the entire array needs to be shifted by one memory location to make space for the new element. This operation takes $O(n)$.
* Removal in the worst case is $O(n)$. See the justification in the previous bullet.
* Appending is $\Omega(n)$ and $O(1)$ amortized (see textbook Section 5.3)

## Representing Multidimensional Data

Lists, tuples, and strings are one-dimensional, i.e., we use a single index to access elements. Many programs involve multi-dimensional data with two, three, or even more dimensions. How can we represent, for example, a 3x3 tic-tac-toe board?

We can represent a chessboard with a two-dimensional array, also called a matrix. We will use two indices $i$ (row) and $j$ (column) to refer to individual cells in the matrix. A common representation for such data is a list of lists. That is, the value is a list of rows, where each row is also a list.

In [18]:
board = [
    [1, 0, 0],
    [0, 0, 0],
    [0, 0, 0]
]

We can then use natural indexing syntax to access individual elements

In [19]:
# Accessing the element at row 0 column 0
print(board[0][0])

1


In [20]:
# Accessing the element at row 0 column 1
print(board[0][1])

0


The construction syntax above assumes we know the dimensions (shape) of the matrix beforehand. What if we wanted to create a matrix dynamically? Suppose we have two variables `col` and `row` and want to create the corresponding matrix.

In [21]:
# Store the dimensions of the matrix in two variables col and row
col = 3
row = 5

def make_matrix(col, row):
    # Try to create the matrix using the following syntax
    matrix = ([0] * col) * row
    return matrix

# Show the result
print(make_matrix(col, row))

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


**That did not work. We ended up with a vector of `col*row` size.**

Let's try something else. Maybe we can use the following syntax to create a list of lists:

In [22]:
onerow = [[0] * 3]
matrix = onerow * 5
matrix
# onerow

[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0]]

In [23]:
def make_matrix(col, row):
    # Try to create the matrix using the following syntax
    onerow = [[0] * col]   #[[0,0,0,0,0]] * row
    matrix = onerow * row
    return matrix

# Show the result
print(make_matrix(col, row))

[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0]]


In [24]:
matrix  = make_matrix(5,3)
for row in matrix: 
    print(row)

[0, 0, 0, 0, 0]
[0, 0, 0, 0, 0]
[0, 0, 0, 0, 0]


That looks better, but it still does not work the way we would expect. Notice what happens when we assign a value to one of the cells and print the result:

In [25]:
matrix[2][0] = 42
print(matrix)

[[42, 0, 0, 0, 0], [42, 0, 0, 0, 0], [42, 0, 0, 0, 0]]


We did not modify just one call; we changed the entire column! This is not what we expected. Why did this happen? Recall that built-in Python lists are implemented with referential arrays. The above syntax creates a matrix where all rows point to the same object:

<img src="https://janakj.org/w3132/images/bad-matrix.png" width=500/>

python automatcially makes shallow copies and those shallow copies all point to the same object since python uses referential arrays for lists and tuples (and compact arrays for strings)

So, we really only have one row. And that row is referenced three times from the enclosing (outer) list, as the above diagram shows.

This kind of data structure is often helpful, but I myself wonder if the syntax `[[0] * col] * row` should work this way. This behavior is unexpected and tends to trip newcomers up. Just remember that creating a multidimensional array using list comprehensions can be tricky. Always double-check that your multi-dimensional data structure looks the way you expect.

The correct way to create a matrix, or an arbitrary multidimensional list, is with the following syntax:

In [26]:
# The correct way to create a matrix
from copy import deepcopy


def make_matrix(col, row):
    return [[0] * col for j in range(row)]

new_row = [0] * col
matrix = []
for j in range(row): 
    my_row = deepcopy(new_row)
    matrix.append(my_row)

matrix = make_matrix(5,3)

# Let's try to modify a single cell within the matrix
matrix[2][0] = 42

# Print the matrix to see the result
print(matrix)

TypeError: 'list' object cannot be interpreted as an integer

### Take-Aways from using Built-in Lists to Represent Multidimensional Data

* Multi-dimensional data can be tricky to implement using the built-in list and tuple types
* The underlying structure is a referential array
* Referential arrays are inefficient for large or highly dimensional data
* Elements can have different data types and sizes (not necessarily what we expect or want)

```python
test = li # is auto-shallow copy
test = li[:] # is a deep copy IFFFF you only use integers, since integers are "primative" immutable values
```


In [35]:
x = [1, 2, 3]
y = x # this is a reference, does not actally create a new object, it just gives like a nickname to the original object x
y[0] = 4
y is x


True

In [ ]:
x = [1, 2, 3]
y = x[:]  ### this creates a faux deep copy. does not share outer items but will share inner items 
y[0] = 4
y is x
print(x, y, y is x)

[1, 2, 3] [4, 2, 3] False


on a nested list, y = x[:] creates a new thing in memory but changing the inner parts of one will change the inner parts of another

In [1]:
li = [1,2,3]

# shallow copy
test = li

# deep copy  
test = li[:]


x = [1,2,3]
y = [4,5,6]
z = [7,8,9]

li = [x,y,z]
test = li 

# shallow copy - inner lists are not duplicated 
li2 = li[:] 


li[0][0] = 0 
li

[[0, 2, 3], [4, 5, 6], [7, 8, 9]]

In [3]:
li2 # notice that 0 also changed here even though we used out (faux) deep copy method

[[0, 2, 3], [4, 5, 6], [7, 8, 9]]

In [ ]:
li is li2 ## is keyword is testing to see if the two objects point to the same place in memory

False

slicing syntax only creates a new copy, so that look deep on a flat list, but on a nestsed list, its very clearly not deep. in both cases y = x[:] returns false because they are not pointing to the same place in memory. 

but if you do like li[0] is li2[0] it will return true because those nested parts are pointing to the same place in memory, but the referential lists in which those same objects in memory lie are different

In [6]:
from copy import deepcopy 

li3 = deepcopy(li)
# this is the only way to make a true deep copy of a nested list

In [48]:
li3[0] is li[0]

False

In [50]:
li2[0] is li[0]

True

### Numpy

In many cases, using the NumPy library to store multidimensional data is better. NumPy is not part of the standard Python library. It needs to be installed with pip or conda and imported:

In [18]:
import numpy as np

Why use NumPy:
* Numpy is a package for numeric computing in Python.
* It provides an efficient data structure for numeric, n-dimensional arrays (ndarray)
* Supports vector and matrix operations.
* Numpy is implemented in C, so it is really fast and efficient.

In [8]:
# Creating multidimensional arrays using numpy.array is straightforward
matrix = numpy.array([[0, 0, 0], [0, 0, 0], [0, 0, 0]])

# See the result
print(matrix)
type(matrix)

[[0 0 0]
 [0 0 0]
 [0 0 0]]


numpy.ndarray

Dynamic matrix creation is also straightforward without the gotchas mentioned above.

The parameter `dtype="uint64"` tells NumPy what data types cells should use. NumPy arrays are homogeneous. Each cell has a maximum value that it can store, determined by the data type.

In [9]:
rows = 4
cols = 3
matrix = numpy.zeros((rows, cols), dtype="uint8") # row x col matrix filled with zeros

print(matrix)
#matrix[2][0] = 42
matrix[2,0] = 255
print(matrix)

[[0 0 0]
 [0 0 0]
 [0 0 0]
 [0 0 0]]
[[  0   0   0]
 [  0   0   0]
 [255   0   0]
 [  0   0   0]]


The basic data type in numpy is the numpy **n-dimensional** array. These can be used to represent vectors (1D) matrices (2D) or tensors (nD).

* 1 dimensions numpy arrays are often used to represent a series of data.
* n-dimensional arrays often represent complete data sets (each column is a type of measurement).

Numpy arrays are very similar to Python lists. Indexing and slicing works the same way (including assingments). *Unlike a Python list, all cells in the same array must contain the same data type.*

Operators don't work the same for lists and arrays and there are many additional methods defined on them.

Some more operations on 1D-arrays:

In [10]:
# u and v are two vectors
u = numpy.array([1,2,3])
v = numpy.array([4,5,6])

In [11]:
# Add vectors element wise
u + v

array([5, 7, 9])

In [12]:
# Add scalar to a vector
u + 4

array([5, 6, 7])

In [13]:
# Multiple vector's elements by a scalar
u * 4

array([ 4,  8, 12])

Many Python built-in operators have been overloaded for numpy arrays to make working with vectors, matrices, and tensors easy.

It's also easy to compute descriptive statistics for a series of data:

In [14]:
u.max(), u.min(), u.mean(), u.std() # maximum, minimum, mean, standard deviation

(np.int64(3), np.int64(1), np.float64(2.0), np.float64(0.816496580927726))

#### Multi-dimensional numpy arrays

In [15]:
# Create directly using literals

m = numpy.array([[1,2,3]])
m.shape

(1, 3)

In [16]:
m = numpy.array([1,2,3])
m.shape

(3,)

In [17]:
# Create a vector and reshape
m = numpy.array(range(27))
m

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26])

In [20]:
n = np.arange(27)
n

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26])

In [21]:
m2 = m.reshape((3,3,3))
m2

array([[[ 0,  1,  2],
        [ 3,  4,  5],
        [ 6,  7,  8]],

       [[ 9, 10, 11],
        [12, 13, 14],
        [15, 16, 17]],

       [[18, 19, 20],
        [21, 22, 23],
        [24, 25, 26]]])

In [22]:
test = numpy.zeros((5,5), dtype = str)

In [26]:
test[0][0] = "hello"
test[0]

array(['h', '', '', '', ''], dtype='<U1')

In [27]:
test

array([['h', '', '', '', ''],
       ['', '', '', '', ''],
       ['', '', '', '', ''],
       ['', '', '', '', ''],
       ['', '', '', '', '']], dtype='<U1')

**Consider NumPy to implement vectors, matrices, or tensors.**

NumPy documentation: https://numpy.org/

You can also find a separate Jupyter notebook in the Files section of CourseWorks dedicated to NumPy. That notebook shows many more useful operations that can be performed on NumPy arrays.

## Stacks

Stacks are one of the most fundamental and useful data structures. Stacks are building blocks of many more sophisticated data structures that come later.

The stack data structure has the following characteristics:
* It is a collection of items (not necessarily of the same types)
* New items can be inserted (pushed) at any time (the size is unbounded)
* Only the most recently inserted item can be accessed (i.e., no indexing)
* Only the most recently inserted item can be removed (popped)
* The most recently inserted item is called the top of the stack

We say a stack follows the last in, first out (LIFO) principle (the most recently pushed item is the one popped next).

<img src="https://janakj.org/w3132/images/stack.png" width=300/>

In [11]:
stack = []
stack.append("A") # use append instead of push
stack.append("B")
stack.append("C")
print(stack.pop())
print(stack.pop())
print(stack.pop())

stack.pop()

C
B
A


IndexError: pop from empty list

### The Stack Abstract Data Type

The abstract data type (ADT) is a contract between the data structure and the programmer. It defines the data and operations (methods) on the data. Since Python is an object-oriented language, we will model abstract data types with classes.

In [51]:
class Empty(Exception):
    '''Exception raised by stack operations on error

    This exception is raised by the stack pop() operation when the stack is empty, i.e.,
    there is nothing to pop.
    '''    
    pass


class Stack:
    '''This class represents the stack abstract data type

    The methods below have empty bodies. We will implement them later in an actual
    stack implementation based on a Python list.
    '''
    
    def push(self, e):
        'Add element e to the top of the stack' 
        pass

    def pop(self):
        'Remove and return the top element from the stack. Raise error if the stack is empty'
        pass

    def top(self): # also known as peek()
        'Return a reference to the top (without removing it)'
        pass

    def is_empty(self):
        'Return True if the stack is empty and false otherwise'
        pass

    def __len__(self):
        'Return the number of elements in the stack'
        pass

In [52]:
s = Stack()
s.push(5)

### Implementing Stacks with Lists (Arrays)

Let's create an actual implementation of the stack ADT. We can use a list to store the stack's item. The operations on the stack can be mapped on the operations on the list as follows (S denotes a stack, L denotes a list):

| Stack        | List        |
|--------------|-------------|
| S.push(e)    | L.append(e) |
| S.pop()      | L.pop()     |
| S.top()      | L\[-1\]     |
| S.is_empty() | len(L) == 0 |
| S.len()      | len(L)      |

In [53]:
class ArrayStack(Stack):
    'A stack implementation that stores elements in a built-in Python list'
    
    def __init__(self):
        'Creates a new stack instance backed by a built-in Python list'
        self._data = list()
    
    def push(self, e):
        'Add element e to the top of the stack'
        self._data.append(e)

    def pop(self):
        'Remove and return the top element from the stack. Raise error if the stack is empty'
        if len(self._data) == 0:
            raise Empty('The stack is empty')

        return self._data.pop()
    
    def top(self):
        'Return a reference to the top (without removing it)'        
        if len(self._data) == 0:
            raise Empty('The stack is empty')
        
        return self._data[-1]

    def is_empty(self):
        'Return True if the stack is empty and false otherwise'
        return len(self._data) == 0

    def __len__(self):
        'Return the number of elements in the stack'
        return len(self._data)

    def __repr__(self): 
        return str(self._data)

Now that we have the stack ADT and an actual implementation, we can try to use it.

In [54]:
s = ArrayStack()
s.push(8)
s.push(2)

# Prints [8, 2]
print(s) # this technically calls print(str(s)) which is print(s.__str__()) 

[8, 2]


In [55]:
# Should return False
s.is_empty()

False

In [56]:
# Should return 2
len(s)

2

In [57]:
result = s.pop()
# Should print False
s.is_empty()

False

In [58]:
result

2

In [59]:
s._data

[8]

In [60]:
s.push(7)
# Should print 7
s.top()

7

In [61]:
s._data

[8, 7]

In [62]:
# Should print 7
s.pop()

7

In [63]:
s.pop()

8

In [64]:
s.pop()

Empty: The stack is empty

### Running Time Analysis
Since all operations on a stack translate to operations on a list, we can analyze the running time of various stack operations by looking at the corresponding list operations. See textbook section 5.3.2 for a description of amortization (gradual reduction of overhead).

| Operation    | Running Time     | Why |
|--------------|------------------|-----|
| S.push(e)    | amortized $O(1)$ | The underlying array may need to be resized as new items are pushed |
| S.pop()      | amortized $O(1)$ | The underlying array may need to be shrunk as items are removed |
| S.top()      | $O(1)$           | Simple indexing of the last element in the list |
| S.is_empty() | $O(1)$           | A list maintains its length in a separate variable which can be accessed in constant-time |
| len(S)       | $O(1)$           | Same as is_empty |

### Example Use Case: Balancing Brackets in an Expression

Let's see an actual use case for a stack. Suppose you are given an expression such as `[(12+x)[0]+{(a+8}]`. Your goal is to write a program that checks if the braces and brackets in the expression are properly balanced (closed). The program should return True if the expression is balanced and False otherwise.

The idea: scan the expression character by character. Whenever we encounter an opening bracket, we push it on the stack. Whenever we encounter a closing bracket, we pop from the stack and check that the closing bracket matches the bracket popped from the stack.

We will develop the function incrementally. We start with a function that simply iterates over the expression's characters.

In [65]:
# This will be the expression to check
expression = '[(12+x)[0]+{(a+8}]'

#expression = '([()])'

#
# This will be our checking function. This version does nothing useful. It only iterates
# over the given expression string character by character.
#
def is_balanced(expr):

    s = ArrayStack()
    
    for c in expr:
        if c in '([{':
            s.push(c)
        elif c == ')':
            if s.top() == '(':
                s.pop()
            else: 
                return False
        elif c == ']':
            if s.top() == '[':
                s.pop()
            else: 
                return False            
        elif c == '}':
            if s.top() == '}':
                s.pop()
            else: 
                return False
    return s.is_empty()

is_balanced(expression)

False

In [66]:
expression = '([()])'

def is_balanced(expr):

    s = ArrayStack()

    open_list = '([{'
    close_list = ')]}'
    
    for c in expr:
        if c in open_list:
            s.push(c)
        elif c in close_list:
            if close_list.index(c) == open_list.index(s.top()):
                s.pop()
            else:
                return False
    return s.is_empty()

is_balanced(expression)

True

Next, we create an ArrayStack instance in our function. Whenever we encounter an opening bracket, we push it on the stack. We need to know what the opening brackets are, so we create a string variable called "opening" in the function that lists them all. We can then check if the encountered character is in the string of opening brackets.

In [67]:
def is_balanced(expr):
    opening = '[({'   
    s = ArrayStack()
    
    for c in expr:
        if c in opening:
            s.push(c)

Let's add code that handles closing brackets. We define another variable called "closing" which will enumerate all closing brackets. For a reason that will be apparent later, the "closing" string must list the closing brackets in the same order as the opening brackets in the "opening" string.

When we encounter a closing bracket, we first check if the stack is not empty. If it is, the expression is unbalanced, and we can return False immediately.

If the stack is not empty, we need to compare the value popped from the stack with the current closing bracket. We cannot compare the two values because they have different ASCII values. However, we said above that we would put the opening and closing brackets at the same index in the two strings. Thus, we can compare their string positions using the string method `index()`.

In [68]:
def is_balanced(expr):
    opening = '[({'
    closing = '])}'
    s = ArrayStack()

    for c in expr:
        if c in opening:
            s.push(c)
        elif c in closing:
            if s.is_empty():
                return False

            v = s.pop()
            if c != v: 
                return False

        

Finally, when we are done iterating over the expression, we must also check that the stack is empty. If it is not, we had more opening than closing brackets and the expression is unbalanced.

In [69]:
def is_balanced(expr):
    opening = '[({'
    closing = '])}'
    s = ArrayStack()

    for c in expr:
        print(c, s._data)
        if c in opening:
            s.push(c)
        elif c in closing:
            if s.is_empty():
                return False

            v = s.pop()
            print(c, opening.index(c),v, closing.index(v))
            if opening.index(c) != closing.index(v):
                return False

    if s.is_empty():
        return True
    else:
        return False

Now we are ready to test the function.

In [70]:
expression = '([()])'

is_balanced(expression)

( []
[ ['(']
( ['(', '[']
) ['(', '[', '(']


ValueError: substring not found

The function correctly returns False because ')' is missing after 8. We can fix the expression and test again:

In [71]:
expression = '[(12+x)[0]+{(a+8)}]'
is_balanced(expression)

[ []
( ['[']
1 ['[', '(']
2 ['[', '(']
+ ['[', '(']
x ['[', '(']
) ['[', '(']


ValueError: substring not found

## Queues

The queue is another fundamental data structure. Queues are similar to stacks but follow the first-in, first-out (FIFO) strategy. Elements enter the queue from the back and are removed from the front.

The queue data structure has the following characteristics:
* It is a collection of items (not necessarily of the same types)
* New item can be inserted (enqueued) at any time (the size is unbounded)
* Only the oldest item can be accessed (i.e., no indexing)
* Only the oldest item can be removed (popped)
* The item that has been on the queue the longest is called the front
* The most recently added item is called the back

<img src="https://janakj.org/w3132/images/queue.png"/>

In [72]:
queue = []
queue.append("A") # use append instead of enqueue
queue.append("B")
queue.append("C") 

queue.pop(0)  #use pop(0) instead of dequeue 

'A'

In [73]:
queue.pop(0)

'B'

In [74]:
queue.pop(0)

'C'

### The Queue Abstract Data Type

In [75]:
class Queue:
    'The queue abstract data type'
    
    def enqueue(self, e):
        'Add element e to the back of the queue'
        pass

    def dequeue(self):
        'Remove and return (first) element from the front of the queue'
        pass

    def first(self):
        'Return a reference to the first element in the queue'
        pass

    def is_empty(self):
        'Return true if the queue is empty'
        pass

    def __len__(self):
        'Return the number of elements in the queue'
        pass

### First (Naive) Array-Based Implementation

Like before, we could store the queue items in a list, i.e., we could create a simple adaption class like ArrayStack. We would then use operations like `list.append(e)` or `list.pop(0)`. The implementation could look as follows.

In [76]:
class ArrayQueue:
    'A queue backed with a built-in Python list'

    def __init__(self):
        'Creates an empty ArrayQueue instance'        
        self._data = list()
    
    def enqueue(self, e):
        'Enqueue element at the back of the queue'
        self._data.append(e)

    def dequeue(self):
        '''Dequeue the element from the front of the queue

        Raise exception if the queue is empty
        '''
        if len(self._data) == 0:
            raise Empty('The queue is empty')

        return self._data.pop(0)
    
    def front(self):
        '''Return a reference to the element at the front

        Raise an exception if the queue is empty
        '''
        if len(self._data) == 0:
            raise Empty('The queue is empty')
        
        return self._data[0]

    def is_empty(self):
        'Return true if the queue is empty'
        return len(self._data) == 0

    def __len__(self):
        'Return the length of the queue'
        return len(self._data)

In [77]:
q = ArrayQueue()
q.enqueue("A")
q.enqueue("B")
q.enqueue("C")

print(q.dequeue())
print(q.dequeue())
print(q.dequeue())

A
B
C


#### Running Time Analysis

Unfortunately, this implementation would be very inefficient. Calling `list.pop(0)`, i.e., removing the first item from the list, causes the rest of the elements to shift, which is always $\Theta(n)$. This is the worst-case behavior. The operations on the above naive implementation have the following asymptotic running times.

| Operation    | Running Time     | Why |
|--------------|------------------|-----|
| enqueue(e)   | amortized $O(1)$ | Optionally enlarge the list and add to the end |
| deque()      | **$\Theta(n)$**  | **Must shift the entire array (list) !!!** |
| first()      | $O(1)$           | Simple indexing |
| is_empty()   | $O(1)$           | Simple variable access |
| len(Q)       | $O(1)$           | Same as above |

### Faster Implementation Using a Fixed-Size Circular List

We can do better than this! We can avoid inefficient list operations such as `list.pop(0)` entirely by using the array (list) in a circular fashion. That is:
  * Allow the front of the queue to move right
  * Allow the contents to wrap around

This approach is summarized in the following diagram.

<img src="https://janakj.org/w3132/images/circular-array.png"/>

How would we actually implement this? Like before, we can store our elements in a list. But let's make the list fixed-size for now, i.e., it will not be enlarged or shrunk. This also means our queue will have a maximum size. We will also keep track of the index of the front element (it will not necessarily be 0). We will also keep the number of elements in the queue in a separate variable. Since we will not necessarily have items starting at index 0, we can no longer rely on the length of the list for queue size.

In [23]:
class Full(Exception):
    'An exception raised when the queue is full'
    pass


class CircularListQueue:
    'A queue implementation using a fixed-size Python list to store elements'
    
    DEFAULT_SIZE = 10  # When creating a new queue, create an empty list of this size

    def __init__(self):
        'Create a new empty queue instance'
        self._data = [ None ] * CircularArrayQueue.DEFAULT_SIZE   # Create a list of DEFAULT_SIZE None values
        self._front = 0    # Initialize the reference to the front of the queue
        self._back = 0     # Initialize the reference to the back of the queue
        self._size = 0     # Logical number of elements. Initially, the queue is empty
    
    def enqueue(self, e):
        'Add an element to the back of the queue'

        # First, check if the queue is full. If it is, raise an exception
        if self._size == len(self._data):
            new_size  = CircularArrayQueue.DEFAULT_SIZE * 2
            new_data = [None] * new_size

            new_back = 0             
            for i in range(len(self)): 
                el =  self.data_[self._front]
                self._front = (self._front + 1) % CircularArrayQueue.DEFAULT_SIZE
                new_data[new_back] = el 
                new_back += 1

            self._data = new_data
            self._front = 0
            self._back = new_back

            
            #raise Full('The queue is full')

        # If it is not, add the item at the back
        self._data[self._back] = e

        # Advance the back pointer modulo the size of the circular list.
        # The modulo operation will make the value wrap around if necessary.
        self._back = (self._back + 1) % CircularArrayQueue.DEFAULT_SIZE

        # And increase the size of the queue
        self._size += 1
    
    def dequeue(self):
        'Remove an element from the front of the queue'

        # First, make sure the queue is not empty
        if len(self._size) == 0:
            raise Empty('The queue is empty')

        # Get the front element
        el = self._data[self._first]

        # Erase the value from the list to help the garbage collector
        self._data[self._front] = None

        # Decrease the size of the queue
        self._size -= 1

        # Update the pointer to the front. Use modulo arithmetic to wrap around
        self._front = (self._front + 1) % CircularArrayQueue.DEFAULT_SIZE

        # Return the previously retrieved element
        return el

    def first(self):
        'Return a reference to the first element in the queue'
        if self._size == 0:
            raise Empty('The queue is empty')

        # Return a reference to the first element in the queue
        return self._data[self._front]

    def is_empty(self):
        'Return true if the queue is empty'
        if self._size == 0:
            return True
        else:
            return False

    def __len__(self):

        return len(self._size)

#### Running Time Analysis

Except for the resizing method, all other methods exhibit a constant number of statements and arithmetic operations. The fixed-size circular array implementation is performant and exhibits constant or amortized constant running times. Voila!

| Operation    | Running Time     | Why |
|--------------|------------------|-----|
| Q.enqueue(e) | amortized $O(1)$ | $O(1)$ |
| Q.deque()    | amortized $O(1)$ | $O(1)$ |
| Q.first()    | $O(1)$           | Simple indexing |
| Q.is_empty() | $O(1)$           | Simple variable access |
| len(Q)       | $O(1)$           | Same as above |



In [80]:
pip install collections

ERROR: Could not find a version that satisfies the requirement collections (from versions: none)
ERROR: No matching distribution found for collections


In [81]:
from collections import dequeue
d = dequeue() #actually implemented using a linked list 

d.append("A")
d.appendLeft("B")
d.pop()
d.popLeft()


ImportError: cannot import name 'dequeue' from 'collections' (/usr/lib/python3.12/collections/__init__.py)

In [79]:
class TwoStackQueue:
    'A queue backed with a built-in Python list'



    def enqueue(self, e):
        'Add element e to the back of the queue'
        pass

    def dequeue(self):
        'Remove and return (first) element from the front of the queue'
        pass

    def first(self):
        'Return a reference to the first element in the queue'
        pass

    def is_empty(self):
        'Return true if the queue is empty'
        pass

    def __len__(self):
        'Return the number of elements in the queue'
        pass